# Phase 2 — Pipeline v2 (Optimized)

**v1 problem:** Python-level loops over groupby results × 396 parts = ~24h.

**v2 fix:** Pure pandas `groupby().agg()` per chunk → `pd.concat` → final
groupby to merge partials. **Zero Python row-iteration.** Expected: 30-60 min.

---

## 0 — Setup

In [2]:
import pandas as pd
import numpy as np
from glob import glob
from scipy import stats
from collections import Counter
import warnings, time, os, gc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
# = "/content/drive/MyDrive/IITD-Tryst-Hackathon/Phase 2"
t0 = time.time()

customers = pd.read_parquet("customers.parquet")
accounts  = pd.read_parquet("accounts.parquet")
linkage   = pd.read_parquet("customer_account_linkage.parquet")
products  = pd.read_parquet("product_details.parquet")
labels    = pd.read_parquet("train_labels.parquet")
test      = pd.read_parquet("test_accounts.parquet")
demographics = pd.read_parquet("demographics.parquet")
acct_extra   = pd.read_parquet("accounts-additional.parquet")
branch_info  = pd.read_parquet("branch.parquet")

for col in ["account_opening_date", "freeze_date", "unfreeze_date",
            "last_mobile_update_date", "last_kyc_date"]:
    if col in accounts.columns:
        accounts[col] = pd.to_datetime(accounts[col], errors="coerce")
for col in ["date_of_birth", "relationship_start_date"]:
    if col in customers.columns:
        customers[col] = pd.to_datetime(customers[col], errors="coerce")
for col in ["address_last_update_date", "passbook_last_update_date"]:
    if col in demographics.columns:
        demographics[col] = pd.to_datetime(demographics[col], errors="coerce")

all_ids = pd.concat([labels[["account_id"]], test[["account_id"]]]).drop_duplicates()
parts = sorted(glob("transactions/batch-*/part_*.parquet"))
print("Loaded statics in {time.time()-t0:.1f}s | {len(all_ids):,} accounts | {len(parts)} txn parts")

Loaded statics in {time.time()-t0:.1f}s | {len(all_ids):,} accounts | {len(parts)} txn parts


## 1 — Chunked Vectorized Feature Extraction

For each part-file we compute **per-account aggregates** using pure pandas
`groupby().agg()` and store the partial DataFrames. At the end we
`pd.concat` all partials and do one final re-aggregation.

In [3]:
# ── Part-level aggregation function ──────────────────────────────────────
def extract_partial(df):
    """Return a per-account summary DataFrame from one txn part. Pure pandas."""
    df = df.copy()
    df["abs_amount"] = df["amount"].abs()
    df["ts"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")
    df["date"] = df["ts"].dt.date
    df["hour"] = df["ts"].dt.hour

    grp = df.groupby("account_id")

    # ── basic aggregates (all vectorized) ────────────────────────────────
    agg = grp.agg(
        txn_count=("abs_amount", "count"),
        total_volume=("abs_amount", "sum"),
        amt_mean=("abs_amount", "mean"),
        amt_std=("abs_amount", "std"),
        amt_max=("abs_amount", "max"),
    ).reset_index()

    # credit / debit volume
    credit = df[df["txn_type"] == "C"].groupby("account_id")["abs_amount"].sum().rename("credit_volume")
    debit  = df[df["txn_type"] == "D"].groupby("account_id")["abs_amount"].sum().rename("debit_volume")
    agg = agg.join(credit, on="account_id").join(debit, on="account_id").fillna({"credit_volume": 0, "debit_volume": 0})

    # R2 — structuring
    near = df[df["abs_amount"].between(45000, 49999)].groupby("account_id").size().rename("near_threshold_count")
    agg = agg.join(near, on="account_id").fillna({"near_threshold_count": 0})

    # R9 — round amounts
    rounds = {1000, 5000, 10000, 50000, 100000}
    rnd = df[df["abs_amount"].isin(rounds)].groupby("account_id").size().rename("round_count")
    agg = agg.join(rnd, on="account_id").fillna({"round_count": 0})

    # H21 — small txns
    small = df[df["abs_amount"] < 500].groupby("account_id").size().rename("small_count")
    agg = agg.join(small, on="account_id").fillna({"small_count": 0})

    # Night
    night = df[df["hour"] < 6].groupby("account_id").size().rename("night_count")
    agg = agg.join(night, on="account_id").fillna({"night_count": 0})

    # H34 — degree (unique counterparties)
    degree = df[df["counterparty_id"].notna()].groupby("account_id")["counterparty_id"].nunique().rename("degree")
    agg = agg.join(degree, on="account_id").fillna({"degree": 0})

    # R4 — credit/debit counterparty counts
    cp_c = df[(df["txn_type"]=="C") & df["counterparty_id"].notna()].groupby("account_id")["counterparty_id"].nunique().rename("credit_cp_n")
    cp_d = df[(df["txn_type"]=="D") & df["counterparty_id"].notna()].groupby("account_id")["counterparty_id"].nunique().rename("debit_cp_n")
    agg = agg.join(cp_c, on="account_id").join(cp_d, on="account_id").fillna({"credit_cp_n": 0, "debit_cp_n": 0})

    # H47 — first large txn timestamp
    large = df[df["abs_amount"] > 50000]
    if len(large) > 0:
        first_large = large.sort_values("ts").groupby("account_id")["ts"].first().rename("first_large_ts")
        agg = agg.join(first_large, on="account_id")
    else:
        agg["first_large_ts"] = pd.NaT

    # Unique MCC count (for entropy proxy)
    mcc_nunique = grp["mcc_code"].nunique().rename("mcc_nunique")
    agg = agg.join(mcc_nunique, on="account_id").fillna({"mcc_nunique": 0})

    # Active days
    active_days = grp["date"].nunique().rename("active_days")
    agg = agg.join(active_days, on="account_id").fillna({"active_days": 0})

    # Gaps — compute within-chunk gap stats
    df_sorted = df.sort_values(["account_id", "ts"])
    df_sorted["gap_h"] = df_sorted.groupby("account_id")["ts"].diff().dt.total_seconds() / 3600
    gap_stats = df_sorted.groupby("account_id")["gap_h"].agg(
        gap_mean="mean", gap_std="std", gap_count="count"
    ).reset_index()
    agg = agg.merge(gap_stats, on="account_id", how="left")

    # R3 — dwell: large credit followed by large debit
    large_txns = df[df["abs_amount"] >= 50000].sort_values(["account_id", "ts"]).copy()
    if len(large_txns) > 0:
        large_txns["next_type"] = large_txns.groupby("account_id")["txn_type"].shift(-1)
        large_txns["next_ts"] = large_txns.groupby("account_id")["ts"].shift(-1)
        pt = large_txns[(large_txns["txn_type"]=="C") & (large_txns["next_type"]=="D")]
        pt = pt.copy()
        pt["dwell_h"] = (pt["next_ts"] - pt["ts"]).dt.total_seconds() / 3600
        dwell_med = pt[pt["dwell_h"] >= 0].groupby("account_id")["dwell_h"].median().rename("median_dwell_hours")
        agg = agg.join(dwell_med, on="account_id")
    else:
        agg["median_dwell_hours"] = np.nan

    return agg

print("✓ extract_partial defined")

✓ extract_partial defined


In [4]:
# ── Process all parts ────────────────────────────────────────────────────
t1 = time.time()
partials = []

for i, p in enumerate(parts):
    df = pd.read_parquet(p)
    partial = extract_partial(df)
    partials.append(partial)
    del df

    if (i+1) % 50 == 0 or i == len(parts)-1:
        elapsed = time.time() - t1
        rate = (i+1) / elapsed * 60
        eta = (len(parts) - i - 1) / (rate/60) / 60 if rate > 0 else 0
        print(f"  [{i+1:3d}/{len(parts)}] {rate:.0f} parts/min | {elapsed/60:.1f}m elapsed | ETA {eta:.0f}m")
    gc.collect()

print(f"\nAll parts processed in {(time.time()-t1)/60:.1f} min")

  [ 50/396] 22 parts/min | 2.2m elapsed | ETA 15m
  [100/396] 22 parts/min | 4.5m elapsed | ETA 13m
  [150/396] 22 parts/min | 6.9m elapsed | ETA 11m
  [200/396] 22 parts/min | 9.2m elapsed | ETA 9m
  [250/396] 22 parts/min | 11.6m elapsed | ETA 7m
  [300/396] 21 parts/min | 14.0m elapsed | ETA 4m
  [350/396] 21 parts/min | 16.3m elapsed | ETA 2m
  [396/396] 21 parts/min | 18.5m elapsed | ETA 0m

All parts processed in 18.5 min


## 2 — Merge Partials into Final Account-Level Features

In [5]:
print("Merging partials...")
all_partials = pd.concat(partials, ignore_index=True)
del partials; gc.collect()

# Re-aggregate across parts (sum counts, weighted means, min of first_large, etc.)
final = all_partials.groupby("account_id").agg(
    txn_count=("txn_count", "sum"),
    total_volume=("total_volume", "sum"),
    amt_max=("amt_max", "max"),
    credit_volume=("credit_volume", "sum"),
    debit_volume=("debit_volume", "sum"),
    near_threshold_count=("near_threshold_count", "sum"),
    round_count=("round_count", "sum"),
    small_count=("small_count", "sum"),
    night_count=("night_count", "sum"),
    degree=("degree", "max"),  # conservative: max across parts
    credit_cp_n=("credit_cp_n", "max"),
    debit_cp_n=("debit_cp_n", "max"),
    first_large_ts=("first_large_ts", "min"),
    mcc_nunique=("mcc_nunique", "max"),
    active_days=("active_days", "sum"),
    gap_mean=("gap_mean", "mean"),
    gap_std=("gap_std", "mean"),
    gap_count=("gap_count", "sum"),
    median_dwell_hours=("median_dwell_hours", "median"),
).reset_index()

# Compute derived features
final["near_threshold_pct"] = final["near_threshold_count"] / final["txn_count"].clip(lower=1)
final["round_amount_pct"]   = final["round_count"] / final["txn_count"].clip(lower=1)
final["night_txn_pct"]      = final["night_count"] / final["txn_count"].clip(lower=1)
final["life_ratio"]         = final["small_count"] / final["txn_count"].clip(lower=1)
final["gap_cv"]             = final["gap_std"] / final["gap_mean"].clip(lower=0.01)
final["fanin_fanout_ratio"] = final["credit_cp_n"] / final["debit_cp_n"].clip(lower=1)

# H47 — days to first large txn
acct_open = dict(zip(accounts["account_id"], accounts["account_opening_date"]))
final["open_date"] = final["account_id"].map(acct_open)
mask = final["first_large_ts"].notna() & final["open_date"].notna()
final.loc[mask, "days_to_first_large"] = (
    final.loc[mask, "first_large_ts"] - final.loc[mask, "open_date"]
).dt.days

# Weighted average for amt_mean — must align weights with non-NaN values
def _wmean(g):
    mask = g["amt_mean"].notna() & (g["txn_count"] > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(g.loc[mask, "amt_mean"], weights=g.loc[mask, "txn_count"])

amt_means = all_partials.groupby("account_id").apply(_wmean)
final["amt_mean"] = final["account_id"].map(amt_means)

# Pooled std — same NaN-safe pattern
def _wstd(g):
    mask = g["amt_std"].notna() & (g["txn_count"] > 0)
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(np.average(g.loc[mask, "amt_std"] ** 2, weights=g.loc[mask, "txn_count"]))

amt_stds = all_partials.groupby("account_id").apply(_wstd)
final["amt_std"] = final["account_id"].map(amt_stds)

del all_partials; gc.collect()

# Rename degree
final.rename(columns={"degree": "degree_centrality"}, inplace=True)

print(f"Final txn features: {final.shape}")
print(final.head(3))

Merging partials...
Final txn features: (160153, 30)
    account_id  txn_count  total_volume    amt_max  credit_volume  \
0  ACCT_000000       2429  6.070175e+07  7964500.0    25140778.05   
1  ACCT_000002       4038  1.135557e+08  7548000.0    41313859.58   
2  ACCT_000003       2450  5.559451e+07  6801288.9    24329631.46   

   debit_volume  near_threshold_count  round_count  small_count  night_count  \
0   35560975.00                  24.0        146.0        946.0         86.0   
1   72241849.90                  23.0        244.0       1662.0        181.0   
2   31264883.29                  20.0        141.0        994.0        115.0   

   degree_centrality  credit_cp_n  debit_cp_n      first_large_ts  \
0                 10         10.0         9.0 2023-09-01 08:51:08   
1                 29         28.0        29.0 2020-07-09 15:42:51   
2                  5          5.0         4.0 2023-10-31 13:36:05   

   mcc_nunique  active_days   gap_mean    gap_std  gap_count  \
0       

## 3 — Trigram Features (Sampled)

In [6]:
# Discover mule-enriched trigrams from train subset
print("Computing trigram features (sampled)...")
t_tri = time.time()

mule_ids = set(labels[labels["is_mule"]==1]["account_id"].head(500))
legit_ids = set(labels[labels["is_mule"]==0]["account_id"].head(500))
sample_ids = mule_ids | legit_ids
mule_lookup = dict(zip(labels["account_id"], labels["is_mule"]))

trigrams_by_class = {0: Counter(), 1: Counter()}

for p in parts[:50]:
    df = pd.read_parquet(p, columns=["account_id", "transaction_timestamp", "txn_type", "channel"])
    df = df[df["account_id"].isin(sample_ids)].sort_values("transaction_timestamp")
    if len(df) == 0: continue
    df["action"] = df["txn_type"].astype(str) + "_" + df["channel"].astype(str)
    for aid, grp in df.groupby("account_id"):
        m = mule_lookup.get(aid, -1)
        if m < 0: continue
        acts = grp["action"].tolist()
        for i in range(len(acts)-2):
            trigrams_by_class[m][f"{acts[i]}|{acts[i+1]}|{acts[i+2]}"] += 1
    del df

tl = sum(trigrams_by_class[0].values()) or 1
tm = sum(trigrams_by_class[1].values()) or 1
enrichment = sorted([
    (t, (trigrams_by_class[1][t]/tm) / max(trigrams_by_class[0].get(t,0)/tl, 1e-8))
    for t in set(trigrams_by_class[0]) | set(trigrams_by_class[1])
    if trigrams_by_class[1].get(t,0) >= 5
], key=lambda x: -x[1])
MULE_TOP10 = set(t for t, _ in enrichment[:10])
print(f"Top-10 mule trigrams ({time.time()-t_tri:.0f}s):")
for t, r in enrichment[:10]:
    print(f"  {t:<55} {r:>8.0f}x")

Computing trigram features (sampled)...
Top-10 mule trigrams (15s):
  D_ATW|D_ATW|D_ATW                                           1708x
  D_NTD|C_UPC|D_MAD                                           1367x
  C_IPM|C_IPM|D_FTD                                           1367x
  D_IPM|D_ATW|D_ATW                                           1025x
  D_P2A|C_IPM|D_NTD                                           1025x
  D_FTD|D_IPM|D_FTD                                           1025x
  C_STD|C_CCL|C_UPD                                           1025x
  D_STD|D_STD|D_STD                                           1025x
  D_ATW|D_UPD|C_FTD                                            854x
  C_MAC|C_IPM|C_MAC                                            854x


In [7]:
# Now compute trigram features for ALL accounts (sampled parts for speed)
print("Computing per-account trigram counts...")
trigram_rows = []
for p in parts[:100]:  # Use 100 parts (25%)
    df = pd.read_parquet(p, columns=["account_id", "transaction_timestamp", "txn_type", "channel"])
    df = df.sort_values(["account_id", "transaction_timestamp"])
    df["action"] = df["txn_type"].astype(str) + "_" + df["channel"].astype(str)
    for aid, grp in df.groupby("account_id"):
        acts = grp["action"].tolist()
        if len(acts) < 3: continue
        tris = [f"{acts[i]}|{acts[i+1]}|{acts[i+2]}" for i in range(len(acts)-2)]
        trigram_rows.append({
            "account_id": aid,
            "mule_trigram_count": sum(1 for t in tris if t in MULE_TOP10),
            "trigram_diversity": len(set(tris)) / len(tris),
        })
    del df
f_trigram = pd.DataFrame(trigram_rows).groupby("account_id").agg(
    mule_trigram_count=("mule_trigram_count", "sum"),
    trigram_diversity=("trigram_diversity", "mean"),
).reset_index()
print(f"Trigram features: {f_trigram.shape} ({time.time()-t_tri:.0f}s)")

Computing per-account trigram counts...
Trigram features: (40074, 3) (221s)


## 4 — transactions_additional (Vectorized)

In [10]:
print("Processing transactions_additional (vectorized)...")
t_add = time.time()

add_parts = sorted(glob("transactions_additional/batch-*/part_*.parquet"))
main_batches = sorted(set(os.path.dirname(p) for p in parts))

ip_partials = []
geo_partials = []
bal_partials = []
subtype_partials = []

for batch_idx, batch_main_dir in enumerate(main_batches):
    batch_name = os.path.basename(batch_main_dir)
    batch_add_dir = os.path.join("transactions_additional", batch_name)
    if not os.path.isdir(batch_add_dir): continue

    # Load just (txn_id, account_id) for this batch
    main_ps = sorted(glob(os.path.join(batch_main_dir, "part_*.parquet")))
    id_map = pd.concat([pd.read_parquet(p, columns=["transaction_id", "account_id"]) for p in main_ps], ignore_index=True)

    # Process additional parts
    add_ps = sorted(glob(os.path.join(batch_add_dir, "part_*.parquet")))
    for ap in add_ps:
        adf = pd.read_parquet(ap).merge(id_map[["transaction_id", "account_id"]], on="transaction_id", how="inner")

        # IP — unique count per account (vectorized)
        if "ip_address" in adf.columns:
            ip_n = adf[adf["ip_address"].notna()].groupby("account_id")["ip_address"].nunique().rename("unique_ip_count").reset_index()
            ip_partials.append(ip_n)

        # Geo — spread per account
        if "latitude" in adf.columns:
            geo = adf[adf["latitude"].notna() & adf["longitude"].notna()]
            if len(geo) > 0:
                geo_agg = geo.groupby("account_id").agg(
                    lat_min=("latitude", "min"), lat_max=("latitude", "max"),
                    lon_min=("longitude", "min"), lon_max=("longitude", "max"),
                    geo_txn_count=("latitude", "count"),
                ).reset_index()
                geo_partials.append(geo_agg)

        # Balance dynamics
        if "balance_after_transaction" in adf.columns:
            bv = adf[adf["balance_after_transaction"].notna()]
            if len(bv) > 0:
                bal_agg = bv.groupby("account_id")["balance_after_transaction"].agg(
                    ["std", "min", "max", "count"]
                ).reset_index()
                bal_agg.columns = ["account_id", "bal_std", "bal_min", "bal_max", "bal_count"]
                bal_partials.append(bal_agg)

        # Sub-type
        if "transaction_sub_type" in adf.columns:
            st = adf[adf["transaction_sub_type"].notna()]
            if len(st) > 0:
                st_counts = st.groupby(["account_id", "transaction_sub_type"]).size().unstack(fill_value=0)
                st_counts["_total"] = st_counts.sum(axis=1)
                subtype_partials.append(st_counts.reset_index())

        del adf
    del id_map; gc.collect()
    print(f"  Batch {batch_idx+1}/{len(main_batches)} ({batch_name}) — {time.time()-t_add:.0f}s")

print(f"Additional processed in {(time.time()-t_add)/60:.1f} min")

Processing transactions_additional (vectorized)...
  Batch 1/4 (batch-1) — 3034s
  Batch 2/4 (batch-2) — 6204s
  Batch 3/4 (batch-3) — 9094s
  Batch 4/4 (batch-4) — 9294s
Additional processed in 154.9 min


In [11]:
# Merge additional partials
f_ip = pd.DataFrame(columns=["account_id", "unique_ip_count"])
if ip_partials:
    f_ip = pd.concat(ip_partials).groupby("account_id")["unique_ip_count"].max().reset_index()
print(f"IP features: {f_ip.shape}")

f_geo = pd.DataFrame(columns=["account_id", "geo_spread_lat", "geo_spread_lon"])
if geo_partials:
    geo_all = pd.concat(geo_partials).groupby("account_id").agg(
        lat_min=("lat_min", "min"), lat_max=("lat_max", "max"),
        lon_min=("lon_min", "min"), lon_max=("lon_max", "max"),
    ).reset_index()
    geo_all["geo_spread_lat"] = geo_all["lat_max"] - geo_all["lat_min"]
    geo_all["geo_spread_lon"] = geo_all["lon_max"] - geo_all["lon_min"]
    f_geo = geo_all[["account_id", "geo_spread_lat", "geo_spread_lon"]]
print(f"Geo features: {f_geo.shape}")

f_bal = pd.DataFrame(columns=["account_id", "balance_std", "balance_min"])
if bal_partials:
    bal_all = pd.concat(bal_partials).groupby("account_id").agg(
        balance_std=("bal_std", "mean"),
        balance_min=("bal_min", "min"),
    ).reset_index()
    f_bal = bal_all
print(f"Balance features: {f_bal.shape}")

f_subtype = pd.DataFrame(columns=["account_id", "cash_txn_pct"])
if subtype_partials:
    st_all = pd.concat(subtype_partials).groupby("account_id").sum().reset_index()
    if "CLT_CASH" in st_all.columns:
        st_all["cash_txn_pct"] = st_all["CLT_CASH"] / st_all["_total"].clip(lower=1)
    else:
        st_all["cash_txn_pct"] = 0
    if "LOAN" in st_all.columns:
        st_all["loan_txn_pct"] = st_all["LOAN"] / st_all["_total"].clip(lower=1)
    else:
        st_all["loan_txn_pct"] = 0
    f_subtype = st_all[["account_id", "cash_txn_pct", "loan_txn_pct"]]
print(f"Subtype features: {f_subtype.shape}")

IP features: (93698, 2)
Geo features: (90929, 3)
Balance features: (98692, 3)
Subtype features: (96117, 3)


## 5 — Static Features

In [12]:
# H9 — Freeze
meta = accounts[["account_id", "freeze_date", "unfreeze_date", "account_status", "daily_avg_balance"]].copy()
meta["has_prior_freeze"] = meta["freeze_date"].notna().astype(int)
meta["has_unfreeze"]     = meta["unfreeze_date"].notna().astype(int)
meta["is_frozen"]        = (meta["account_status"] == "frozen").astype(int)

# H38 — Residual
meta = meta.merge(final[["account_id", "total_volume"]], on="account_id", how="left")
meta["residual_ratio"] = meta["daily_avg_balance"].abs() / meta["total_volume"].clip(lower=1)
meta["is_tiny_residual"] = (meta["residual_ratio"] < 0.001).astype(int)

# R8 — Mobile update flag
meta["has_mobile_update"] = accounts["last_mobile_update_date"].notna().astype(int).values

# Aadhaar
cust = linkage[["account_id","customer_id"]].merge(customers[["customer_id","aadhaar_available"]], on="customer_id", how="left")
cust["aadhaar_missing"] = (cust["aadhaar_available"].isna() | (cust["aadhaar_available"]=="N")).astype(int)
meta = meta.merge(cust[["account_id","aadhaar_missing"]], on="account_id", how="left")
f_meta = meta[["account_id","has_prior_freeze","has_unfreeze","is_frozen",
               "residual_ratio","is_tiny_residual","has_mobile_update","aadhaar_missing"]]

# R12 — Branch rate
ab = accounts[["account_id","branch_code"]].merge(labels[["account_id","is_mule"]], on="account_id", how="inner")
gr = ab["is_mule"].mean()
bs = ab.groupby("branch_code").agg(n=("is_mule","count"), pos=("is_mule","sum")).reset_index()
bs["branch_mule_rate"] = (bs["pos"] + 10*gr) / (bs["n"] + 10)
f_branch_rate = accounts[["account_id","branch_code"]].merge(bs[["branch_code","branch_mule_rate"]], on="branch_code", how="left")
f_branch_rate["branch_mule_rate"].fillna(gr, inplace=True)
f_branch_rate = f_branch_rate[["account_id", "branch_mule_rate"]]

# Scheme
scheme = pd.get_dummies(acct_extra["scheme_code"], prefix="scheme").astype(int)
scheme["account_id"] = acct_extra["account_id"].values
scheme["is_pmjdy"] = (acct_extra["scheme_code"]=="PMJDY").astype(int).values

# Demographics
demo = demographics.merge(linkage[["customer_id","account_id"]], on="customer_id", how="inner")
demo["is_joint_account"] = (demo["joint_account_flag"]=="Y").astype(int)
demo["is_nri"] = (demo["nri_flag"]=="Y").astype(int)
demo["is_male"] = (demo["gender"]=="M").astype(int)
demo["shared_phone_count"] = demo.groupby("phone_number")["account_id"].transform("count")
demo["shared_address_count"] = demo.groupby("address")["account_id"].transform("count")
f_demo = demo[["account_id","is_joint_account","is_nri","is_male","shared_phone_count","shared_address_count"]]

# Branch enrichment
br = accounts[["account_id","branch_code"]].merge(branch_info, on="branch_code", how="left")
br["is_urban"] = (br["branch_type"]=="urban").astype(int)
br["is_rural"] = (br["branch_type"]=="rural").astype(int)
for c in ["branch_employee_count","branch_turnover","branch_asset_size"]:
    br[c] = pd.to_numeric(br[c], errors="coerce")
f_branch = br[["account_id","is_urban","is_rural","branch_employee_count","branch_turnover","branch_asset_size"]]

print(f"Static features ready: meta={f_meta.shape}, branch_rate={f_branch_rate.shape}, scheme={scheme.shape}, demo={f_demo.shape}, branch={f_branch.shape}")

Static features ready: meta=(160153, 8), branch_rate=(160153, 2), scheme=(160153, 9), demo=(160153, 6), branch=(160153, 6)


## 6 — Merge Everything

In [13]:
features = all_ids.copy()
for name, df in [
    ("TxnCore",     final), ("Trigram", f_trigram),
    ("IP",          f_ip), ("Geo", f_geo), ("Balance", f_bal), ("Subtype", f_subtype),
    ("Meta",        f_meta), ("BranchRate", f_branch_rate),
    ("Scheme",      scheme), ("Demo", f_demo), ("BranchInfo", f_branch),
]:
    before = features.shape[1]
    features = features.merge(df, on="account_id", how="left")
    print(f"  {name:<12} +{features.shape[1]-before:>2} cols → {features.shape[1]} total")

# H42 interactions
features["degree_x_night"]   = features["degree_centrality"] * features["night_txn_pct"]
features["gapcv_x_degree"]   = features["gap_cv"] * features["degree_centrality"]
features["gapcv_x_near50k"]  = features["gap_cv"] * features["near_threshold_pct"]
features["degree_x_near50k"] = features["degree_centrality"] * features["near_threshold_pct"]

# R10 composite
sc = [c for c in ["near_threshold_pct","round_amount_pct","gap_cv","degree_centrality",
      "mule_trigram_count","branch_mule_rate","has_prior_freeze"] if c in features.columns]
for c in sc:
    m,s = features[c].mean(), features[c].std()
    features[c+"_z"] = (features[c]-m)/s if s>0 else 0
features["composite_score"] = features[[c+"_z" for c in sc]].sum(axis=1)
features.drop(columns=[c+"_z" for c in sc], inplace=True)

print(f"\nFinal: {features.shape}")

  TxnCore      +29 cols → 30 total
  Trigram      + 2 cols → 32 total
  IP           + 1 cols → 33 total
  Geo          + 2 cols → 35 total
  Balance      + 2 cols → 37 total
  Subtype      + 2 cols → 39 total
  Meta         + 7 cols → 46 total
  BranchRate   + 1 cols → 47 total
  Scheme       + 8 cols → 55 total
  Demo         + 5 cols → 60 total
  BranchInfo   + 5 cols → 65 total

Final: (160153, 70)


## 7 — Quality Check & Save

In [14]:
nan_pct = features.drop(columns=["account_id"]).isna().mean().sort_values(ascending=False)
print(f"Features with >50% NaN: {(nan_pct>0.5).sum()}")
print(f"Features with 0% NaN: {(nan_pct==0).sum()}")
print(f"Total features: {features.shape[1]-1}")

train_f = features.merge(labels[["account_id","is_mule"]], on="account_id", how="inner")
num_cols = train_f.select_dtypes(include=[np.number]).columns
corr = train_f[num_cols].corr()["is_mule"].drop("is_mule", errors="ignore").abs().sort_values(ascending=False)
print(f"\nTop 15 by |corr| with is_mule:")
print(corr.head(15).to_string())

Features with >50% NaN: 2
Features with 0% NaN: 52
Total features: 69

Top 15 by |corr| with is_mule:
composite_score      0.389296
has_prior_freeze     0.353215
branch_mule_rate     0.329540
is_frozen            0.316125
degree_x_night       0.294170
degree_centrality    0.282143
credit_cp_n          0.212083
debit_cp_n           0.191806
gapcv_x_degree       0.183322
has_unfreeze         0.155179
branch_turnover      0.152783
life_ratio           0.122370
degree_x_near50k     0.098460
trigram_diversity    0.093595
gap_cv               0.092277


In [15]:
train_out = features[features["account_id"].isin(labels["account_id"])].merge(labels[["account_id","is_mule"]], on="account_id")
test_out = features[features["account_id"].isin(test["account_id"])]
train_out.to_csv("features_train_p2.csv", index=False)
test_out.to_csv("features_test_p2.csv", index=False)
print(f"\n✅ Train: {train_out.shape} (mule={train_out['is_mule'].mean():.4f})")
print(f"✅ Test:  {test_out.shape}")
print(f"Total: {time.time()-t0:.0f}s = {(time.time()-t0)/60:.0f} min")


✅ Train: (96091, 71) (mule=0.0279)
✅ Test:  (64062, 70)
Total: 26235s = 437 min
